# Peter TTS Auto (Kaggle headless)
Pushed by GitHub Actions via `kaggle kernels push`. Starts GPU, generates all clips, auto-stops.
No Gradio URL needed. Output goes to `/kaggle/working/audio/`.


In [ ]:
# Cell 1: deps (Kaggle already has torch; add the rest)
!pip install -q transformers accelerate librosa soundfile einops omegaconf sentencepiece protobuf datasets


In [ ]:
# Cell 2: fetch repo (sparse, /tmp only so kernel output stays clean) + parse script
import os
SCRIPT_SEL_DEFAULT = "config/scripts/EPISODE_01.txt"
REPO_DIR = "/tmp/pvrepo"
!rm -rf /tmp/pvrepo peter-video-maker-github
!git clone --depth 1 --filter=blob:none --sparse https://github.com/0xSatwik/peter-video-maker-github.git /tmp/pvrepo
!git -C /tmp/pvrepo sparse-checkout set --no-cone config/scripts assets/peter-voice.mp3 assets/Stewies-voice.mp3 assets/perter10seonds.wav

SCRIPT = os.path.join(REPO_DIR, SCRIPT_SEL_DEFAULT)
print("SCRIPT:", SCRIPT, os.path.exists(SCRIPT))

import re
def preprocess_text(t):
    t = re.sub(r'\s+', ' ', t)
    t = re.sub(r'([.,!?])([A-Za-z])', r'\1 \2', t)
    return t.strip()

def parse_script(path):
    lines = []
    print("Parsing:", path)
    with open(path, encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if line.lower().startswith("format:") or line.lower().startswith("family guy"):
                continue
            parts = line.split("|")
            if len(parts) == 3:
                sp = parts[0].strip().lower()
                tx = preprocess_text(parts[2].strip())
                if not tx.endswith((".", "!", "?", '"', "'")):
                    tx += "."
                lines.append({"speaker": sp, "text": tx})
    return lines

VOICE_REFS = {
    "peter": [os.path.join(REPO_DIR, "assets/peter-voice.mp3"), os.path.join(REPO_DIR, "assets/perter10seonds.wav")],
    "stewie": [os.path.join(REPO_DIR, "assets/Stewies-voice.mp3")],
}
for k, ps in VOICE_REFS.items():
    print(k, [p for p in ps if os.path.exists(p)] or "MISSING")


In [ ]:
# Cell 3: load MOSS-TTS 1.7B (same as Colab) + batch generate to /kaggle/working/audio
# NOTE: kernel auto-terminates when this cell finishes -> no idle GPU burn.
import gc, json, time, warnings
import torch
from transformers import AutoModel, AutoProcessor, GenerationConfig
warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("device:", device)
assert device == "cuda", "GPU not attached! Enable GPU in kernel-metadata."

class DelayGenerationConfig(GenerationConfig):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.layers = kwargs.get("layers", [{} for _ in range(32)])
        self.do_samples = kwargs.get("do_samples", None)
        self.n_vq_for_inference = 32

# Proven settings (matches scripts/generate_audio.py HIGH_QUALITY)
HQ = dict(max_new_tokens=2500, speed=1, text_temp=0.7, text_top_p=0.9, text_top_k=40,
            audio_temp=0.7, audio_top_p=0.85, audio_top_k=40, audio_rep_pen=1.15, n_vq=24)

print("Loading processor...")
processor = AutoProcessor.from_pretrained("OpenMOSS-Team/MOSS-TTS-Local-Transformer", trust_remote_code=True)
processor.audio_tokenizer = processor.audio_tokenizer.to(device)
torch.cuda.empty_cache(); gc.collect()
print("Loading model (~13GB, 5-8 min first run)...")
model = AutoModel.from_pretrained("OpenMOSS-Team/MOSS-TTS-Local-Transformer", trust_remote_code=True,
    attn_implementation="sdpa", torch_dtype=dtype, low_cpu_mem_usage=True).to(device)
model.eval()
print(f"Model ready. VRAM: {torch.cuda.memory_allocated()/1024**3:.2f}GB")

lines = parse_script(SCRIPT)
print(f"Lines: {len(lines)}")
OUT = "/kaggle/working/audio"
os.makedirs(OUT, exist_ok=True)

def resolve_ref(sp):
    for p in VOICE_REFS.get(sp, []):
        if os.path.exists(p):
            return p
    return None

ok, fail = 0, 0
meta = []
t0 = time.time()
for i, ln in enumerate(lines):
    sp, text = ln["speaker"], ln["text"]
    out = f"{OUT}/{sp}_{i:03d}.wav"
    ref = resolve_ref(sp)
    if ref is None and sp in VOICE_REFS:
        print(f"SKIP {i}: missing ref for {sp}"); fail += 1
        meta.append({"index": i, "speaker": sp, "text": text, "audio_file": out, "exists": False})
        continue
    try:
        s = time.time()
        convs = [[processor.build_user_message(text=text, reference=[ref])]] if ref else [[processor.build_user_message(text=text)]]
        batch = processor(convs, mode="generation")
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        tt, at = HQ["text_temp"], HQ["audio_temp"]
        if tt == 1.0: tt = 1.001
        if at == 1.0: at = 1.001
        g = DelayGenerationConfig()
        g.pad_token_id = processor.tokenizer.pad_token_id
        g.eos_token_id = 151653
        g.max_new_tokens = HQ["max_new_tokens"]
        g.use_cache = True; g.do_sample = True; g.num_beams = 1
        g.n_vq_for_inference = HQ["n_vq"]
        g.do_samples = [True] * (HQ["n_vq"] + 1)
        g.layers = [{"repetition_penalty": 1.0, "temperature": tt, "top_p": HQ["text_top_p"], "top_k": HQ["text_top_k"]}]
        g.layers += [{"repetition_penalty": HQ["audio_rep_pen"], "temperature": at, "top_p": HQ["audio_top_p"], "top_k": HQ["audio_top_k"]} for _ in range(HQ["n_vq"])]
        with torch.no_grad():
            out_ids = model.generate(input_ids, attention_mask=attn, generation_config=g)
        decoded = processor.decode(out_ids)
        audio = decoded[0].audio_codes_list[0]
        import torchaudio
        torchaudio.save(out, audio.unsqueeze(0).cpu(), processor.model_config.sampling_rate)
        print(f"OK [{i+1}/{len(lines)}] {sp} {time.time()-s:.1f}s -> {out}")
        ok += 1; meta.append({"index": i, "speaker": sp, "text": text, "audio_file": out, "exists": True})
    except Exception as e:
        print(f"FAIL [{i+1}/{len(lines)}] {e}"); fail += 1
        meta.append({"index": i, "speaker": sp, "text": text, "audio_file": out, "exists": False})

with open(f"{OUT}/metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)
print(f"DONE: {ok} ok, {fail} fail in {(time.time()-t0)/60:.1f} min. Files in {OUT}")
print(os.listdir(OUT))
